# PEARL - Google Colab Runner

This notebook connects your Google Drive and runs the PEARL project directly from it, ensuring all your data, checkpoints, and results are safely stored in your Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup the Project Workspace
Assuming you've uploaded `pearl.zip` to your main Google Drive directory (`My Drive`), this will unzip it into a folder called `pearl_project`.

In [ ]:
# Unzip the project (Only run this cell ONCE to extract your project)
!unzip -q /content/drive/MyDrive/pearl.zip -d /content/drive/MyDrive/pearl_project

In [ ]:
# Change directory into the newly extracted project folder
import os

# NOTE: Adjust this path if your extracted folder structure is different
os.chdir('/content/drive/MyDrive/pearl_project/pearl')
!pwd

## 3. Install Requirements
*Note: Google Colab already has PyTorch installed and optimized. This step just installs the missing libraries like `timm`, `albumentations`, etc.*

In [ ]:
!pip install -r requirements.txt

## 4. Run the Pipeline Steps

In [ ]:
# 1. Preprocess
!python scripts/preprocess.py \
    --config configs/preprocessing/full_ad.yaml \
    --data_dir ./data \
    --split_seed 42

In [ ]:
# 2. Train a single model
!python scripts/train.py \
    --model configs/model/efficientnet_b4.yaml \
    --preprocessing configs/preprocessing/full_ad.yaml \
    --experiment configs/experiment/best_model_xai.yaml

In [ ]:
# 3. Run the full 54-run sweep
!python scripts/sweep.py --experiment configs/experiment/sweep_all_54.yaml

In [ ]:
# 4. Hyperparameter tuning
!python scripts/tune.py \
    --experiment configs/experiment/tune_best.yaml \
    --study_name efficientnet_b4_full_ad \
    --storage sqlite:///results/tuning/optuna.db

In [ ]:
# 5. Calibration analysis
!python scripts/run_calibration.py \
    --model configs/model/efficientnet_b4.yaml \
    --preprocessing configs/preprocessing/full_ad.yaml \
    --checkpoint results/checkpoints/efficientnet_b4__full_ad.pt

In [ ]:
# 6. Uncertainty quantification
!python scripts/run_uncertainty.py \
    --model configs/model/efficientnet_b4.yaml \
    --preprocessing configs/preprocessing/full_ad.yaml \
    --checkpoint results/checkpoints/efficientnet_b4__full_ad.pt \
    --mc_passes 50

In [ ]:
# 7. Explainability (XAI)
!python scripts/run_xai.py \
    --model configs/model/efficientnet_b4.yaml \
    --preprocessing configs/preprocessing/full_ad.yaml \
    --checkpoint results/checkpoints/efficientnet_b4__full_ad.pt \
    --methods gradcam lrp shap \
    --n_samples 20